# (선택) 글로 사진 만들기 — 집에서 해 보는 생성 모델

**군-학 연계 프로그램 범용 교육 7차시 · 선택 노트북.** 수업의 탐지기 실습(실습 3 · 4)에 쓴 AI 사진 5장(`docs/탐지용사진/`)은 이 노트북으로 2026-10-12에 Colab T4에서 만든 것입니다. 집에서 같은 것을 직접 만들어 봅니다.

## 사용 규칙
1. **위에서 아래로** ▶. 바꿔 봐도 되는 줄은 `# <-- 바꿔 보세요` 줄뿐.
2. **GPU 필수** — 런타임 → 런타임 유형 변경 → T4 GPU → 저장.
3. **사람 · 얼굴 · 실존 인물 · 로고를 넣지 않습니다.** 사물 · 풍경 · 문서만. 만든 사진은 확인이 끝나면 지웁니다.

| 셀 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 1 | 준비(도구 설치 · 생성 모델 SDXL-Turbo 약 7GB 내려받기 · 진짜 사진 2장) | 약 3분 |
| 2 | 글로 사진 4장 만들기 | 약 10초 |
| 3 | 문서 사진 1장 더 · 진짜 사진과 함께 zip 내려받기 | 약 10초 |

## 셀 1. 준비 — 도구를 설치하고 이미지 생성 모델을 내려받습니다
▶를 누르고 "준비 완료"가 나올 때까지 기다립니다(약 3분). 생성 모델은 공개 모델 **SDXL-Turbo**(Stability AI, 연구·비상업용 무료)입니다. 진짜 사진 2장은 5차시에 쓴 Ultralytics 공개 사진입니다.

In [ ]:
# 셀 1. 준비 — 도구 설치 + 이미지 생성 모델(약 7GB) 내려받기 + 진짜 사진 2장 (약 3분)
import subprocess, sys, os, time, zipfile, shutil, urllib.request
t0 = time.time()
# 버전 고정(2026-10-12 실측): diffusers 0.40 은 transformers 4.57.6 이 내리는 huggingface-hub 0.36 과 안 맞아 파이프라인 로드가 멈춘다. → 둘 다 고정.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.36.0", "transformers==4.57.6", "accelerate"], check=True)
print(f"설치 끝 ({(time.time() - t0) / 60:.1f}분)")

import torch
assert torch.cuda.is_available(), "GPU 없음 — 메뉴 [런타임 > 런타임 유형 변경 > T4 GPU > 저장] 뒤 이 셀을 다시 실행하세요."
print("GPU:", torch.cuda.get_device_name(0))

from diffusers import AutoPipelineForText2Image
GEN_ID = "stabilityai/sdxl-turbo"
pipe = AutoPipelineForText2Image.from_pretrained(GEN_ID, torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.set_progress_bar_config(disable=True)
print("생성 모델:", GEN_ID)

def fetch(url, path):
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (lecture notebook)"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            shutil.copyfileobj(r, f)

os.makedirs("made", exist_ok=True)
fetch("https://ultralytics.com/images/bus.jpg", "made/real_1.jpg")        # 진짜 사진 (카메라로 찍은 것)
fetch("https://ultralytics.com/images/zidane.jpg", "made/real_2.jpg")
from PIL import Image
from IPython.display import display
print(f"준비 완료 ({(time.time() - t0) / 60:.1f}분)")

## 셀 2. 만들기 — 글로 사진 4장을 만듭니다
`PROMPT`(영어)를 바꿔 가며 만듭니다. **사람 · 얼굴은 넣지 않습니다.** 장당 1~2초. 결과는 `made/ai_1.png ~ ai_4.png`에 저장됩니다.

In [ ]:
# 셀 2. 만들기 — 글로 사진 4장 (장당 1~2초)
PROMPT = "a photo of a wooden office desk with a laptop, a coffee mug and documents, window light, realistic"   # <-- 바꿔 보세요 (영어, 사람 없이)
N = 4            # <-- 바꿔 보세요: 몇 장
STEPS = 2        # <-- 바꿔 보세요: 1~4 (클수록 조금 더 정교, 느림)

t0 = time.time()
gens = [torch.Generator("cuda").manual_seed(i) for i in range(N)]
images = pipe(prompt=[PROMPT] * N, num_inference_steps=STEPS, guidance_scale=0.0, generator=gens).images
for i, im in enumerate(images, 1):
    im.save(f"made/ai_{i}.png")
print(f"{N}장, {time.time() - t0:.1f}초. made/ai_1.png ~ ai_{N}.png")

grid = Image.new("RGB", (512 * 2, 512 * ((N + 1) // 2)), "white")
for i, im in enumerate(images):
    grid.paste(im.resize((512, 512)), ((i % 2) * 512, (i // 2) * 512))
display(grid.resize((grid.width // 2, grid.height // 2)))

## 셀 3. 한 장 더 — 문서 사진을 만들고, 진짜 사진과 함께 zip으로 내려받습니다
"서명 있는 계약서"를 글로 만들어 봅니다. 그다음 `만든사진.zip`(AI 5장 + 진짜 2장)을 내려받아 탐지 사이트 · Content Credentials 확인 사이트에 올려 봅니다(prompts/04 · 05).

In [ ]:
# 셀 3. 한 장 더 — 문서 사진 + zip 내려받기
PROMPT2 = "a scanned photo of a signed paper contract on a desk, handwritten signature at the bottom, realistic"   # <-- 바꿔 보세요
image = pipe(prompt=PROMPT2, num_inference_steps=STEPS, guidance_scale=0.0,
             generator=torch.Generator("cuda").manual_seed(7)).images[0]
image.save("made/ai_doc.png")
display(image.resize((320, 320)))

with zipfile.ZipFile("만든사진.zip", "w") as z:
    for f in sorted(os.listdir("made")):
        z.write(os.path.join("made", f), f)
print("만든사진.zip:", sorted(os.listdir("made")))
print("탐지 사이트와 Content Credentials 확인 사이트에 ai_*.png 와 real_*.jpg 를 하나씩 올려 본다.")

from google.colab import files
files.download("만든사진.zip")